# Sultan Hassan Mosque-Madrasa — SDXL LoRA (Task-Compliant)
Matches the brief exactly:
- Per-image **captions** (not one flat prompt) — required deliverable + graded criterion
- Trigger word `sltnhsn` is auto-prepended to every caption
- Generates all **5 required test prompts x 4 fixed seeds** at the end, including the no-trigger-word control prompt
- Saves your **training settings** and a **notes template** as files, ready to drop into the deliverable folder
- Uses SDXL (an explicitly allowed "suitable alternative" to Flux per the brief) with the fp16-fix VAE, xformers if available, and plain AdamW (bitsandbytes 8-bit Adam is broken on current Colab images)

**Before you start:** your dataset should be a zip where each image has a matching `.txt` caption file with the same base name, e.g.:
```
img001.jpg
img001.txt   <- "entrance portal with muqarnas, massive stone facade, vertical window bays"
img002.jpg
img002.txt   <- "courtyard with four iwans, sunlight on stone"
```
Do NOT put the trigger word in your .txt files yourself — this notebook adds `sltnhsn,` to the front of every caption automatically so it's consistent across all images.

In [ ]:
# Step 0: Clean slate — remove bitsandbytes if a previous session left it installed.
# It is broken on current Colab images (Python 3.13 + cu128) and will crash diffusers on import.
!pip uninstall -y bitsandbytes -q
print("If bitsandbytes was previously imported in this session, go to Runtime > Restart session now, then continue from Step 1.")

In [ ]:
# Step 1: Install pinned, tested-compatible libraries. Torch is left alone —
# reinstalling/pinning it breaks against Colab's current CUDA driver.
import torch
print("Colab's preinstalled torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

!pip install -q -U \
  diffusers==0.31.0 \
  transformers==4.46.3 \
  accelerate==1.1.1 \
  peft==0.13.2 \
  datasets==3.1.0 \
  xformers \
  wandb

print("If this cell errors on xformers, that's OK — rerun without pinning: !pip install -q -U xformers")

In [ ]:
# Step 1b: Confirm xformers actually imports against this torch build.
try:
    import xformers
    print("xformers OK:", xformers.__version__)
    USE_XFORMERS = True
except Exception as e:
    print("xformers failed to import, training will run without it (slower/more VRAM):", e)
    USE_XFORMERS = False

In [ ]:
# Step 2: Upload your dataset zip (25-40 images + one matching .txt caption per image)
import os
from google.colab import files

print("Upload your zipped dataset (images + matching .txt caption files):")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

!rm -rf dataset_raw
!unzip -q -o {zip_name} -d dataset_raw

IMG_EXTS = {".png", ".jpg", ".jpeg", ".webp"}

def find_content_dir(root):
    for dirpath, _, filenames in os.walk(root):
        if any(os.path.splitext(f)[1].lower() in IMG_EXTS for f in filenames):
            return dirpath
    return None

src_dir = find_content_dir("dataset_raw")
if src_dir is None:
    raise RuntimeError("No images found anywhere under ./dataset_raw — check your zip structure!")

images = sorted([f for f in os.listdir(src_dir) if os.path.splitext(f)[1].lower() in IMG_EXTS])
print(f"Found {len(images)} images in {src_dir}")

if not (25 <= len(images) <= 40):
    print(f"WARNING: brief asks for 25-40 images, you have {len(images)}. Continuing anyway.")

missing_captions = [f for f in images if not os.path.exists(os.path.join(src_dir, os.path.splitext(f)[0] + ".txt"))]
if missing_captions:
    raise RuntimeError(f"{len(missing_captions)} images have no matching .txt caption file, e.g. {missing_captions[:5]}. "
                        "Every image needs a caption per the brief — add captions and re-upload.")

print("All images have matching captions. Good to proceed.")

In [ ]:
# Step 3: Resize/crop consistently, prepend trigger word to captions, build metadata.jsonl
# (the format diffusers' text-to-image LoRA trainer expects: {"file_name": ..., "text": ...})
import json
from PIL import Image

TRIGGER = "sltnhsn"
TARGET_RES = 1024  # matches --resolution used in training below

os.makedirs("dataset", exist_ok=True)

def center_crop_resize(img, size):
    w, h = img.size
    scale = size / min(w, h)
    new_w, new_h = round(w * scale), round(h * scale)
    img = img.resize((new_w, new_h), Image.LANCZOS)
    left = (new_w - size) // 2
    top = (new_h - size) // 2
    return img.crop((left, top, left + size, top + size))

metadata_rows = []
skipped = []
for fname in images:
    base = os.path.splitext(fname)[0]
    src_img_path = os.path.join(src_dir, fname)
    src_cap_path = os.path.join(src_dir, base + ".txt")

    try:
        img = Image.open(src_img_path).convert("RGB")
    except Exception as e:
        skipped.append((fname, str(e)))
        continue

    if min(img.size) < TARGET_RES:
        print(f"NOTE: {fname} is smaller than {TARGET_RES}px on its short side ({img.size}); "
              "brief asks for >=1024px source images. Upscaling to fit — consider replacing this image.")

    out_img = center_crop_resize(img, TARGET_RES)
    out_name = base + ".jpg"
    out_img.save(os.path.join("dataset", out_name), quality=95)

    with open(src_cap_path, "r", encoding="utf-8") as f:
        caption = f.read().strip()
    if not caption.lower().startswith(TRIGGER):
        caption = f"{TRIGGER}, {caption}"

    metadata_rows.append({"file_name": out_name, "text": caption})

with open(os.path.join("dataset", "metadata.jsonl"), "w", encoding="utf-8") as f:
    for row in metadata_rows:
        f.write(json.dumps(row) + "\n")

print(f"Prepared {len(metadata_rows)} image/caption pairs in ./dataset")
if skipped:
    print(f"Skipped {len(skipped)} unreadable files: {skipped}")

print("\nSample captions:")
for row in metadata_rows[:3]:
    print(" -", row["file_name"], "->", row["text"])

In [ ]:
# Step 4: Download the official diffusers SDXL LoRA (text-to-image, caption-aware) training script
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/v0.31.0/examples/text_to_image/train_text_to_image_lora_sdxl.py
print("Script downloaded.")

In [ ]:
# Step 5: Configure accelerate non-interactively (single T4 GPU, fp16)
from accelerate.utils import write_basic_config
write_basic_config(mixed_precision="fp16")
print("accelerate config written.")

In [ ]:
# Step 6: Record training settings NOW (before launch) — this is a required deliverable.
TRAINING_SETTINGS = {
    "base_model": "stabilityai/stable-diffusion-xl-base-1.0",
    "vae": "madebyollin/sdxl-vae-fp16-fix",
    "method": "LoRA (rank 32) via diffusers train_text_to_image_lora_sdxl.py",
    "trigger_word": "sltnhsn",
    "resolution": 1024,
    "train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "learning_rate": "1e-4",
    "lr_scheduler": "constant",
    "max_train_steps": 1500,
    "checkpointing_steps": 250,
    "seed": 42,
    "mixed_precision": "fp16",
    "rank": 32,
    "optimizer": "AdamW (bitsandbytes 8-bit Adam not usable on current Colab image — see notes)",
    "num_training_images": len(metadata_rows),
}

with open("training_settings.json", "w") as f:
    json.dump(TRAINING_SETTINGS, f, indent=2)

print(json.dumps(TRAINING_SETTINGS, indent=2))
print("\nSaved to training_settings.json — include this file in your deliverables.")

In [ ]:
# Step 7: Train! (~1-1.5 hours on a T4 for 1500 steps on ~30 images)
# rank=32 gives more style capacity than the default rank=4 for an architectural style like this.
# If you OOM: lower --resolution to 768, or lower --train_batch_size / rank.
xformers_flag = "--enable_xformers_memory_efficient_attention" if USE_XFORMERS else ""

!accelerate launch train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --train_data_dir="dataset" \
  --caption_column="text" \
  --output_dir="sultan_hassan_sdxl_lora" \
  --mixed_precision="fp16" \
  --resolution=1024 \
  --center_crop \
  --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --rank=32 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=1500 \
  --checkpointing_steps=250 \
  --seed=42 \
  --gradient_checkpointing \
  {xformers_flag}

In [ ]:
# Step 8: Save your trained LoRA (Download it!)
!zip -r -q sultan_hassan_sdxl_lora.zip sultan_hassan_sdxl_lora
from google.colab import files
files.download('sultan_hassan_sdxl_lora.zip')

In [ ]:
# Step 9: Generate the REQUIRED test grid — 5 prompts x 4 fixed seeds each = 20 images.
# Includes the no-trigger-word control prompt to check for style bleeding.
from diffusers import DiffusionPipeline, AutoencoderKL
import torch, os, json

vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae,
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")
pipe.load_lora_weights("sultan_hassan_sdxl_lora")

PROMPTS = [
    "sltnhsn, elevation",
    "sltnhsn, entrance gates",
    "sltnhsn, a courtyard with a central fountain at sunset",
    "sltnhsn, a mosque at night",
    "a modern glass office tower",  # control prompt: NO trigger word, checks for style bleeding
]
SEEDS = [1000, 1001, 1002, 1003]  # fixed seeds, required by the brief

os.makedirs("test_images", exist_ok=True)
manifest = []

for p_idx, prompt in enumerate(PROMPTS):
    safe_name = prompt.replace(", ", "_").replace(" ", "-")[:40]
    for seed in SEEDS:
        generator = torch.Generator(device="cuda").manual_seed(seed)
        image = pipe(prompt, num_inference_steps=30, generator=generator).images[0]
        fname = f"test_images/{p_idx:02d}_{safe_name}_seed{seed}.png"
        image.save(fname)
        manifest.append({"prompt": prompt, "seed": seed, "file": fname})
        print("Saved", fname)

with open("test_images/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\nDone: {len(manifest)} images across {len(PROMPTS)} prompts x {len(SEEDS)} seeds.")

In [ ]:
# Step 10: Zip and download the test images + manifest
!zip -r -q test_images.zip test_images
from google.colab import files
files.download('test_images.zip')

## Step 11: Write your deliverable note (half page)
The brief asks for a short note explaining your **data choices** and how you reduced **hallucination and style bleeding**. Use this as a starting skeleton — fill in the specifics of what you actually did, don't just submit it unedited:

---
**Data choices.** I collected N images of the Sultan Hassan Mosque-Madrasa covering exterior facades, the entrance portal, the courtyard and its four iwans, minarets, the dome, and interior ornament/muqarnas/lamp details, favoring [describe your source mix, e.g. daytime + a few night shots for the night-mosque prompt]. I excluded or cropped out any frame where the neighbouring Al-Rifa'i Mosque dominated, since its similar massing risks being learned as part of the "sltnhsn" style. All images were center-cropped and resized to 1024px.

**Captioning.** Each image has a caption describing only what's distinctive in that frame (e.g. "entrance portal with muqarnas, massive stone facade" vs. "courtyard with four iwans"), with the trigger word `sltnhsn` prepended uniformly. This keeps the trigger word bound to the *style*, not to any one fixed composition, which is what lets prompts like "sltnhsn, a mosque at night" generalize rather than just reproducing a training photo.

**Reducing hallucination / style bleeding.** [Describe: LoRA rank chosen, step count vs. dataset size to avoid overfitting/underfitting, whether you visually reviewed checkpoints at 250-step intervals, and what the control prompt without `sltnhsn` showed — ideally a generic glass tower with no Mamluk ornament, confirming the style didn't leak into the base model's general vocabulary.]

---